# RQ1 dataset qualification and chaptering pilot (diagnostic)

Outside the 22-node official manifest. Source probes are replayed **offline** from the bounded, logged cache written by `experiments.scripts.rq1_candidate_ledger`. The pilot uses only `validation` units, `evidence_status=partial` and the `diagnostic` namespace. A passing cell is not an RQ1 result; the conclusion is the GO/BLOCKED ledger at the end.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, subprocess
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
assert (ROOT / "experiments/scripts").is_dir(), "run from the project root"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
QUAL = ROOT / "experiments/outputs/diagnostic/rq1_qualification"
RUN = ROOT / "experiments/outputs/diagnostic/rq1_pilot_runs" / STAMP
RUN.mkdir(parents=True)
os.environ.update(PYTHONIOENCODING="utf-8")
commands = []
def command(args, accepted=(0,), timeout=900):
    result = subprocess.run(args, cwd=ROOT, text=True, encoding="utf-8", errors="replace",
                            capture_output=True, timeout=timeout)
    entry = dict(command=args, exit_code=result.returncode, accepted=result.returncode in accepted,
                 stdout=result.stdout[-4000:], stderr=result.stderr[-2000:])
    commands.append(entry)
    (RUN / "commands.json").write_text(json.dumps(commands, indent=2), encoding="utf-8")
    print("COMMAND", " ".join(map(str, args)), "EXIT", result.returncode)
    print(result.stdout[-3000:])
    return result
print("Run directory:", RUN)


Run directory: C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\outputs\diagnostic\rq1_pilot_runs\20260927T065316Z


## 1. Focused contract tests
GT provenance, split sealing (final_test first in manifest, smoke truncation), same-unit features, ±3 s collar edges, empty predictions, checkpoint identity, manifest edges, dataset contracts, endpoint eligibility.

In [2]:
command([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
         "tests/test_rq1_chaptering_contract.py", "tests/test_dataset_contracts.py",
         "tests/test_endpoint_eligibility.py", "tests/test_chapter_metrics.py"])
assert commands[-1]["accepted"], "focused tests failed"


COMMAND C:\Users\hung\AppData\Local\Programs\Python\Python311\python.exe -m pytest -q -p no:cacheprovider tests/test_rq1_chaptering_contract.py tests/test_dataset_contracts.py tests/test_endpoint_eligibility.py tests/test_chapter_metrics.py EXIT 0
........................................................................ [ 72%]
...........................                                              [100%]
99 passed in 11.88s



## 2. Candidate ledger (offline replay of bounded probes)

In [3]:
command([sys.executable, "-m", "experiments.scripts.rq1_candidate_ledger",
         "--output-dir", str(QUAL), "--offline"])
assert commands[-1]["accepted"]
ledger = json.loads((QUAL / "candidate_ledger.json").read_text(encoding="utf-8"))
assert ledger["network"]["offline_replay"] and ledger["network"]["new_requests_this_invocation"] == 0
rows = []
for r in ledger["records"]:
    e = r["evidence"]
    rows.append(dict(source=r["source_id"], course=r["course_id"], lecture=r["lecture_id"],
                     split=e.get("official_split"), boundaries=len(e.get("chapter_boundaries_sec") or []),
                     media_duration=(e.get("media_probe") or {}).get("duration_sec"),
                     rights=r["rights_status"], access=r["access_status"], excluded=r["exclusion_reason"]))
table = pd.DataFrame(rows)
display(table.groupby(["source", table.excluded.isna().rename("unexcluded")]).size().rename("units").reset_index())
display(table[table.source != "ytseg"])
display(Markdown("YTSeg summary"))
display(pd.json_normalize(ledger["ytseg_summary"]).T)
display(pd.json_normalize(ledger["decision"]).T)


COMMAND C:\Users\hung\AppData\Local\Programs\Python\Python311\python.exe -m experiments.scripts.rq1_candidate_ledger --output-dir C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\outputs\diagnostic\rq1_qualification --offline EXIT 0
{
  "avlectures_archive": "blocked_access",
  "ocw_timed_toc": {
    "lectures_found": 35,
    "lectures_without_exclusion": 30,
    "media_verified_lectures": 5,
    "independent_source_courses": [
      "18-01-single-variable-calculus-fall-2006"
    ],
    "status": "diagnostic_pilot_candidate (baseline-only; single source course)"
  },
  "ytseg": {
    "multimodal_video": "blocked_rights",
    "text_audio": "candidate (creator chapters; official splits)",
    "official_split_source_disjoint": false
  },
  "course_disjoint_learned_study": "blocked_data",
  "official_multimodal_rq1": "BLOCKED",
  "official_blockers": [
    "AVLectures archive requires authenticated access (blocked_access)",
    "only on

,source,unexcluded,units
0,avlectures_cws,False,1
1,mit_ocw_timed_toc,False,5
2,mit_ocw_timed_toc,True,30
3,ytseg,False,80


,source,course,lecture,split,boundaries,media_duration,rights,access,excluded
0,avlectures_cws,*,*,NaN,0,NaN,conflicting_repository_license_and_unknown_vid...,blocked_access,archive download requires an authenticated ind...
1,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec39,NaN,8,2353.179864,cc_by_nc_sa: ocw clip pages 4.0 + archive.org ...,media_header_decoded,NaN
2,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec15,NaN,8,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,overlapping_or_invalid_offsets
3,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec19,NaN,7,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,NaN
4,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec14,NaN,5,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,NaN
5,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec04,NaN,7,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,NaN
6,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec05,NaN,7,2940.133878,cc_by_nc_sa: ocw clip pages 4.0 + archive.org ...,media_header_decoded,NaN
7,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec16,NaN,5,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,NaN
8,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,18-01-single-variable-calculus-fall-2006:lec01,NaN,8,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,NaN
9,mit_ocw_timed_toc,18-01-single-variable-calculus-fall-2006,youtube:9v25gg2qJYE,NaN,11,NaN,cc_by_nc_sa_4.0_page_notice_per_clip_no_reserv...,media_url_published,overlapping_or_invalid_offsets;ambiguous_sourc...


YTSeg summary

,0
api_status,200
revision,4ee1c1fccc5a9e118ff0918f3df1e61aadca8970
last_modified,2026-07-20T13:16:43.000Z
gated,False
license_tags,[license:cc-by-nc-sa-4.0]
has_video_files,False
has_download_videos_script,True
sample_channel_overlap_validation_test,"[channel:UC-CSyyi47VX1lD9zyeABW3w, channel:UC4..."
sampled_units,80
split_totals.validation,1447


,0
avlectures_archive,blocked_access
course_disjoint_learned_study,blocked_data
official_multimodal_rq1,BLOCKED
official_blockers,[AVLectures archive requires authenticated acc...
ocw_media_probe_sample,[18-01-single-variable-calculus-fall-2006:lec2...
ocw_timed_toc.lectures_found,35
ocw_timed_toc.lectures_without_exclusion,30
ocw_timed_toc.media_verified_lectures,5
ocw_timed_toc.independent_source_courses,[18-01-single-variable-calculus-fall-2006]
ocw_timed_toc.status,diagnostic_pilot_candidate (baseline-only; sin...


## 3. Boundary/media alignment audit
Published offsets vs decoded media duration for the pre-declared sample. Offsets are never edited.

In [4]:
audit = []
for r in ledger["records"]:
    e = r["evidence"]
    if not e.get("media_probe_sample"):
        continue
    audit.append(dict(lecture=r["lecture_id"], clips=e["timed_clip_count"],
                      first_clip_start=e["first_start_sec"], last_clip_end=e["last_end_sec"],
                      media_duration=e["media_probe"].get("duration_sec"),
                      end_minus_media=round(e["last_end_sec"] - (e["media_probe"].get("duration_sec") or 0), 3),
                      gaps=e["gap_count"], max_gap=e["max_gap_sec"],
                      media_license=(e.get("media_item_rights") or {}).get("licenseurl"),
                      excluded=r["exclusion_reason"], titles=e["clip_titles"]))
display(pd.DataFrame(audit))


,lecture,clips,first_clip_start,last_clip_end,media_duration,end_minus_media,gaps,max_gap,media_license,excluded,titles
0,18-01-single-variable-calculus-fall-2006:lec39,9,20.0,2354.0,2353.179864,0.820,0,0.0,http://creativecommons.org/licenses/by-nc-sa/3.0/,NaN,"[Clip 1: Review of Taylor's Series, Clip 2: Ta..."
1,18-01-single-variable-calculus-fall-2006:lec05,8,20.0,2941.0,2940.133878,0.866,0,0.0,http://creativecommons.org/licenses/by-nc-sa/3.0/,NaN,[Clip 1: Introduction to Implicit Differentiat...
2,18-01-single-variable-calculus-fall-2006:lec23,7,20.0,2915.0,2915.961905,-0.962,0,0.0,http://creativecommons.org/licenses/by-nc-sa/3.0/,NaN,"[Clip 1: Average Value, Clip 2: Average Height..."
3,18-01-single-variable-calculus-fall-2006:lec30,8,20.0,3083.0,3083.679637,-0.680,0,0.0,http://creativecommons.org/licenses/by-nc-sa/3.0/,NaN,"[Clip 1: Partial Fractions – Big Example, Clip..."
4,18-01-single-variable-calculus-fall-2006:lec03,8,20.0,2995.0,2994.491791,0.508,1,59.0,http://creativecommons.org/licenses/by-nc-sa/3.0/,NaN,"[Clip 1: Introduction to Differentiation, Clip..."
5,18-01-single-variable-calculus-fall-2006:lec11,4,20.0,3104.0,2981.720816,122.279,0,0.0,http://creativecommons.org/licenses/by-nc-sa/3.0/,clip_end_after_media_end,"[Clip 3: Graphing a Rational Function, Clip 1:..."


## 4. Validation-only pilot of the 04a/04b runners
Fresh diagnostic root; all units `validation`; single course, so no final_test exists to leak.

In [5]:
PILOT = RUN / "artifacts"
command([sys.executable, "-m", "experiments.scripts.rq1_diagnostic_pilot",
         "--ledger", str(QUAL / "candidate_ledger.json"), "--artifact-root", str(PILOT),
         "--cache-dir", str(QUAL)], accepted=(0, 1))
pilot = json.loads((PILOT / "pilot_summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: pilot[k] for k in ("unit_ids", "splits", "exit_codes", "evidence_status")}, indent=2))


COMMAND C:\Users\hung\AppData\Local\Programs\Python\Python311\python.exe -m experiments.scripts.rq1_diagnostic_pilot --ledger C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\outputs\diagnostic\rq1_qualification\candidate_ledger.json --artifact-root C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\outputs\diagnostic\rq1_pilot_runs\20260927T065316Z\artifacts --cache-dir C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\outputs\diagnostic\rq1_qualification EXIT 0
dia_probe_sample"
    },
    {
      "lecture_id": "18-01-single-variable-calculus-fall-2006:lec33",
      "reason": "not_in_media_probe_sample"
    },
    {
      "lecture_id": "18-01-single-variable-calculus-fall-2006:lec09",
      "reason": "not_in_media_probe_sample"
    },
    {
      "lecture_id": "18-01-single-variable-calculus-fall-2006:lec27",
      "reason": "n

## 5. Receipt verification and split audit

In [6]:
from benchmarks.scripts.execute_all_notebooks import load_manifest, validate_produced_artifacts
entries = {e["id"]: e for e in load_manifest()["notebooks"]}
assert len(entries) == 22
NS = PILOT / "diagnostic" / "full"
receipt_errors = {}
for nid in ("04a_chaptering_benchmark", "04b_chaptering_modality_ablation"):
    run = json.loads((NS / "runs" / f"{nid}.run.json").read_text(encoding="utf-8"))
    ctx = {k: run[k] for k in ("run_id", "status", "source_sha256", "config_sha256", "dependency_state_sha256")}
    receipt_errors[nid] = validate_produced_artifacts(entries[nid], PILOT, "full", "diagnostic", ctx)
    assert run["evaluation_mode"] == "selection" and run["status"] == "partial"
print("receipt errors:", receipt_errors)
def records(aid):
    return json.loads((NS / "payloads" / f"{aid}.json").read_text(encoding="utf-8"))["records"]
frozen_units = records("dataset.frozen_manifest")[0]["unit_records"]
unit_split = {u["unit_id"]: u["split"] for u in frozen_units}
seen = {aid: sorted({r["unit_id"] for r in records(aid)}) for aid in (
    "chapter.input_features", "chapter.raw_predictions", "chapter.per_unit_metrics",
    "chapter.ablation_predictions", "chapter.ablation_metrics")}
non_validation = {aid: [u for u in ids if unit_split[u] != "validation"] for aid, ids in seen.items()}
print("non-validation units per artifact:", non_validation)
assert not any(non_validation.values())
assert len({tuple(v) for v in seen.values()}) == 1, "artifacts do not cover the same units"


receipt errors: {'04a_chaptering_benchmark': [], '04b_chaptering_modality_ablation': []}
non-validation units per artifact: {'chapter.input_features': [], 'chapter.raw_predictions': [], 'chapter.per_unit_metrics': [], 'chapter.ablation_predictions': [], 'chapter.ablation_metrics': []}


## 6. Pilot outputs (diagnostic, not RQ1 performance)

In [7]:
display(pd.DataFrame([{"candidate": r["candidate"], "units": r["unit_count"],
                       "empty_predictions": r["empty_prediction_count"], **r["metrics"]}
                      for r in records("chapter.aggregate")]))
display(pd.DataFrame(records("chapter.candidate_status")))
display(pd.DataFrame(records("chapter.ablation_status")))
display(pd.DataFrame([{"unit": p["unit_id"], "candidate": p["candidate"],
                       "predicted": p["predictions"]["chapter_boundaries_sec"][:8],
                       "reference": p["predictions"]["reference_boundaries_sec"]}
                      for p in records("chapter.raw_predictions")]))
display(pd.json_normalize(records("chapter.frozen_configuration")[0]).T)


,candidate,units,empty_predictions,collar_f1_3s,collar_f1_5s,collar_f1_10s,pk,window_diff
0,fixed_window_300s,5,0,0.000000,0.000000,0.050000,0.476739,0.480115
1,lexical_texttiling,5,0,0.029889,0.049434,0.074659,0.565150,1.000000


,candidate,status,family,required_modalities,unit_count,reason,units_missing,checkpoint_path
0,fixed_window_300s,run,heuristic,[],5.0,NaN,NaN,NaN
1,lexical_texttiling,run,heuristic,[transcript],5.0,NaN,NaN,NaN
2,multimodal_scene_aligned,not_run,heuristic,"[transcript, scene]",5.0,missing_modality,5.0,NaN
3,C5_cross_attention,not_run,learned,NaN,NaN,checkpoint_provenance_missing: no train-only d...,NaN,C:\Users\hung\Documents\GitHub\multimodal-lect...
4,C6_late_fusion,not_run,learned,NaN,NaN,checkpoint_missing,NaN,C:\Users\hung\Documents\GitHub\multimodal-lect...


,ablation,status,required_modalities,reason,units_missing
0,duration_only,run,[],NaN,NaN
1,transcript,run,[transcript],NaN,NaN
2,transcript+ocr,not_run,"[transcript, ocr]",missing_modality,5.0
3,transcript+scene,not_run,"[transcript, scene]",missing_modality,5.0
4,transcript+visual,not_run,"[transcript, visual]",missing_modality,5.0
5,transcript+acoustic,not_run,"[transcript, acoustic]",missing_modality,5.0
6,all_modalities,not_run,"[transcript, ocr, scene, visual, acoustic]",missing_modality,5.0


,unit,candidate,predicted,reference
0,18-01-single-variable-calculus-fall-2006:lec03,fixed_window_300s,"[300.0, 600.0, 900.0, 1200.0, 1500.0, 1800.0, ...","[135.0, 510.0, 908.0, 1216.0, 1440.0, 2110.0, ..."
1,18-01-single-variable-calculus-fall-2006:lec03,lexical_texttiling,"[4.782, 19.128, 33.474, 47.82, 62.166, 76.512,...","[135.0, 510.0, 908.0, 1216.0, 1440.0, 2110.0, ..."
2,18-01-single-variable-calculus-fall-2006:lec05,fixed_window_300s,"[300.0, 600.0, 900.0, 1200.0, 1500.0, 1800.0, ...","[101.0, 537.0, 833.0, 1184.0, 1743.0, 2153.0, ..."
3,18-01-single-variable-calculus-fall-2006:lec05,lexical_texttiling,"[5.135, 20.539, 35.943, 51.347, 66.751, 82.155...","[101.0, 537.0, 833.0, 1184.0, 1743.0, 2153.0, ..."
4,18-01-single-variable-calculus-fall-2006:lec23,fixed_window_300s,"[300.0, 600.0, 900.0, 1200.0, 1500.0, 1800.0, ...","[371.0, 543.0, 1104.0, 1375.0, 2253.0, 2733.0]"
5,18-01-single-variable-calculus-fall-2006:lec23,lexical_texttiling,"[5.118, 20.474, 35.829, 51.184, 71.658, 87.013...","[371.0, 543.0, 1104.0, 1375.0, 2253.0, 2733.0]"
6,18-01-single-variable-calculus-fall-2006:lec30,fixed_window_300s,"[300.0, 600.0, 900.0, 1200.0, 1500.0, 1800.0, ...","[1100.0, 1273.0, 1474.0, 1646.0, 1991.0, 2414...."
7,18-01-single-variable-calculus-fall-2006:lec30,lexical_texttiling,"[5.034, 20.135, 35.236, 50.338, 65.439, 85.574...","[1100.0, 1273.0, 1474.0, 1646.0, 1991.0, 2414...."
8,18-01-single-variable-calculus-fall-2006:lec39,fixed_window_300s,"[300.0, 600.0, 900.0, 1200.0, 1500.0, 1800.0, ...","[622.0, 825.0, 1316.0, 1475.0, 1600.0, 1855.0,..."
9,18-01-single-variable-calculus-fall-2006:lec39,lexical_texttiling,"[4.676, 18.706, 32.735, 46.764, 60.794, 74.823...","[622.0, 825.0, 1316.0, 1475.0, 1600.0, 1855.0,..."


,0
frozen_algorithm,lexical_texttiling
frozen_modality_variant,transcript
selection_split,validation
selection_metric,collar_f1_3s
selection_collar_f1_3s,0.029889
variant_table,"[{'variant': 'duration_only', 'collar_f1_3s': ..."
selection_unit_ids,[18-01-single-variable-calculus-fall-2006:lec0...
final_test_evaluated,False


## 7. GO/BLOCKED readiness ledger

In [8]:
decision = ledger["decision"]
ocw = decision["ocw_timed_toc"]
checks = [
    ("contract_tests", "PASS" if commands[0]["accepted"] else "FAIL", "focused suites"),
    ("dataset_access", "BLOCKED", "AVLectures archive: " + decision["avlectures_archive"]
        + "; OCW timed-TOC media URLs published and headers decoded for "
        + str(ocw["media_verified_lectures"]) + " lectures"),
    ("rights", "PENDING_REVIEW", "OCW pages CC BY-NC-SA 4.0 + archive.org item 3.0, no reserved marking; "
        "YTSeg video blocked_rights; human rights review not done"),
    ("independent_gt", "PARTIAL", "OCW publisher timed-TOC (editorial excerpt offsets) aligned for "
        + str(ocw["media_verified_lectures"]) + "/6 sampled lectures; YTSeg creator chapters (editorial)"),
    ("course_disjoint_splits", "BLOCKED", decision["course_disjoint_learned_study"]
        + ": OCW cohort = " + ", ".join(ocw["independent_source_courses"])
        + "; YTSeg val/test share channels"),
    ("d2_receipt", "BLOCKED", "no official D2 freeze; pilot uses diagnostic 00c-equivalent artifacts"),
    ("same_unit_features", "PASS" if not any(non_validation.values()) else "FAIL",
        "chapter.input_features hash-bound per unit; transcript only (OCR/scene/visual/acoustic blocked_compute)"),
    ("candidate_models", "BLOCKED", "; ".join(f"{s['candidate']}={s['status']}"
        for s in records("chapter.candidate_status") if s["status"] != "run")),
    ("notebook_runs", "PASS" if pilot["exit_codes"] == {"04a": 0, "04b": 0}
        and not any(receipt_errors.values()) else "FAIL", "04a/04b runners + executor receipt validation"),
    ("compute", "BLOCKED", "per-unit OCR/scene/visual/acoustic extraction not run (full media not downloaded)"),
]
ready = pd.DataFrame(checks, columns=["gate", "status", "evidence"])
display(ready)
overall = "GO" if all(s == "PASS" for _, s, _ in checks) else "BLOCKED"
out = {"generated_at": STAMP, "overall_official_rq1": overall, "gates": ready.to_dict("records"),
       "ledger": str(QUAL / "candidate_ledger.json"), "pilot_root": str(PILOT),
       "claim": "diagnostic only; no RQ1 hypothesis, final-test or full-DAG claim"}
(RUN / "readiness_ledger.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
print("OVERALL official RQ1:", overall)
print("Readiness ledger:", RUN / "readiness_ledger.json")


,gate,status,evidence
0,contract_tests,PASS,focused suites
1,dataset_access,BLOCKED,AVLectures archive: blocked_access; OCW timed-...
2,rights,PENDING_REVIEW,OCW pages CC BY-NC-SA 4.0 + archive.org item 3...
3,independent_gt,PARTIAL,OCW publisher timed-TOC (editorial excerpt off...
4,course_disjoint_splits,BLOCKED,blocked_data: OCW cohort = 18-01-single-variab...
5,d2_receipt,BLOCKED,no official D2 freeze; pilot uses diagnostic 0...
6,same_unit_features,PASS,chapter.input_features hash-bound per unit; tr...
7,candidate_models,BLOCKED,multimodal_scene_aligned=not_run; C5_cross_att...
8,notebook_runs,PASS,04a/04b runners + executor receipt validation
9,compute,BLOCKED,per-unit OCR/scene/visual/acoustic extraction ...


OVERALL official RQ1: BLOCKED
Readiness ledger: C:\Users\hung\Documents\GitHub\multimodal-lecture-summarizer\multimodal-lecture-summarizer\experiments\outputs\diagnostic\rq1_pilot_runs\20260927T065316Z\readiness_ledger.json
